# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1, 2: 1 is sum of empty elements, 2 is we filled the none element with the mean (1+3/2=2) so the final series is (1,2,3)
2. ["a","a","b"], 2. nunique is an amount of unique elements, array we got is because of methods .strip() (removes spaces) and .lower() (turns all letters to lowercase)
3. {"k": [1, 2]}
4. 1005
5. [9, 10]

## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
# 1
import pandas as pd
def clean(path):
    df = pd.read_csv(path)
    df['amount'] = df['amount'].astype(str).str.replace(' ', '')
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
    print(f"После конвертации сумм в числа: {len(df)} строк")
    сity_mapping = {
        'алматы': 'Алматы', 'алмата': 'Алматы', 'almaty': 'Алматы',
        'астана': 'Астана', 'astana': 'Астана',
        'караганда': 'Караганда', 'karaganda': 'Караганда', 'крг': 'Караганда',
        'шымкент': 'Шымкент', 'shymkent': 'Шымкент',
        'актау': 'Актау', 'aktau': 'Актау',
        'атырау': 'Атырау', 'atyrau': 'Атырау'
    }
    df['city'] = df['city'].astype(str).str.strip().str.lower().map(city_mapping).fillna(df['city'])
    print(f"После стандартизации городов: {len(df)} строк")
    df = df.drop_duplicates()
    print(f"После удаления дубликатов: {len(df)} строк")
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    print(f"После преобразования дат: {len(df)} строк")
    df = df[df['city'] != 'TEST']
    df = df[df['customer_id'] != 'TEST']
    print(f"После исключения TEST-записей: {len(df)} строк")
    
    df = df.dropna(subset=['amount', 'city'])
    print(f"После удаления строк с NaN в 'amount' и 'city': {len(df)} строк")
    assert df['amount'].isna().sum() == 0, "Ошибка: в 'amount' остались пропуски!"
    assert df['city'].isna().sum() == 0, "Ошибка: в 'city' остались пропуски!"
    assert df['order_id'].is_unique, "Ошибка: 'order_id' содержит дубликаты!"
    print("\nВсе проверки успешно пройдены!")
    
    return df
    


In [5]:
display(df.dtypes)
df.info()

order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount               str
delivery_days    float64
status               str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   str    
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(1), int64(1), str(6)
memory usage: 705.5 KB


In [7]:
df['amount'] = df['amount'].astype(str).str.replace(' ', '')
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df['amount'] = df['amount'].astype('Int64')
print(df['amount'].dtype)
display(df[['amount']].head())

Int64


,amount
0,8070
1,3630
2,9120
3,9350
4,11010


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [9]:
#2
df_paid = df_clean[df_clean['status'] == 'оплачен'].copy()
df_paid['month'] = df_paid['order_date'].dt.month

total_sep = df_paid[df_paid['month'] == 9]['amount'].sum()
total_oct = df_paid[df_paid['month'] == 10]['amount'].sum()
total_pct_change = (total_oct - total_sep) / total_sep * 100

print("ОБЩАЯ ВЫРУЧКА")
print(f"Сентябрь: {total_sep:,.0f} KZT")
print(f"Октябрь: {total_oct:,.0f} KZT")
print(f"Изменение: {total_pct_change:.2f}%")
print("\nВЫРУЧКА ПО ГОРОДАМ")

cities = df_paid['city'].unique()
city_results = []

for city in cities:
    city_df = df_paid[df_paid['city'] == city]
    sep_val = city_df[city_df['month'] == 9]['amount'].sum()
    oct_val = city_df[city_df['month'] == 10]['amount'].sum()
    pct_change = (oct_val - sep_val) / sep_val * 100 if sep_val > 0 else 0
    city_results.append({
        'City': city,
        'September': sep_val,
        'October': oct_val,
        'Change %': pct_change
    })
res_df = pd.DataFrame(city_results).sort_values(by='Change %')
display(res_df)

ОБЩАЯ ВЫРУЧКА
Сентябрь: 35,243,410 KZT
Октябрь: 34,419,520 KZT
Изменение: -2.34%

ВЫРУЧКА ПО ГОРОДАМ


,City,September,October,Change %
3,Шымкент,4558650.0,4044010.0,-11.289307
2,Караганда,3648520.0,3371680.0,-7.587734
1,Астана,9034220.0,8850530.0,-2.033269
5,Актобе,2576960.0,2540420.0,-1.417950
0,Алматы,14501070.0,14643020.0,0.978893
4,Атырау,923990.0,969860.0,4.964339


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [10]:
# 3
import pandas as pd

df_raw = pd.read_csv('orders.csv')

def get_revenue_drop(temp_df):
    t = temp_df.copy()
    t['amount_clean'] = t['amount'].astype(str).str.replace(' ', '')
    t['amount_clean'] = pd.to_numeric(t['amount_clean'], errors='coerce').fillna(0)
    t['date_clean'] = pd.to_datetime(t['order_date'], errors='coerce')
    t['month'] = t['date_clean'].dt.month

    paid = t[t['status'] == 'оплачен']

    sep = paid[paid['month'] == 9]['amount_clean'].sum()
    oct_val = paid[paid['month'] == 10]['amount_clean'].sum()

    if sep == 0:
        return 0
    return (oct_val - sep) / sep * 100

steps = []

curr_df = df_raw.copy()
steps.append({"Шаг": "0. Исходные данные", "Строк осталось": len(curr_df), "Убрано строк": 0, "Падение выручки %": get_revenue_drop(curr_df)})

curr_df['amount'] = curr_df['amount'].astype(str).str.replace(' ', '')
curr_df['amount'] = pd.to_numeric(curr_df['amount'], errors='coerce')
steps.append({"Шаг": "1. Числовой формат amount", "Строк осталось": len(curr_df), "Убрано строк": 0, "Падение выручки %": get_revenue_drop(curr_df)})

city_mapping = {
    'алматы': 'Алматы', 'алмата': 'Алматы', 'almaty': 'Алматы',
    'астана': 'Астана', 'astana': 'Астана',
    'караганда': 'Караганда', 'karaganda': 'Караганда', 'крг': 'Караганда',
    'шымкент': 'Шымкент', 'shymkent': 'Шымкент',
    'актау': 'Актау', 'aktau': 'Актау',
    'атырау': 'Атырау', 'atyrau': 'Атырау'
}
curr_df['city'] = curr_df['city'].astype(str).str.strip().str.lower().map(city_mapping).fillna(curr_df['city'])
steps.append({"Шаг": "2. Стандартизация городов", "Строк осталось": len(curr_df), "Убрано строк": 0, "Падение выручки %": get_revenue_drop(curr_df)})

before_dup = len(curr_df)
curr_df = curr_df.drop_duplicates()
removed_dup = before_dup - len(curr_df)
steps.append({"Шаг": "3. Удаление дубликатов", "Строк осталось": len(curr_df), "Убрано строк": removed_dup, "Падение выручки %": get_revenue_drop(curr_df)})

curr_df['order_date'] = pd.to_datetime(curr_df['order_date'], errors='coerce')
steps.append({"Шаг": "4. Преобразование дат", "Строк осталось": len(curr_df), "Убрано строк": 0, "Падение выручки %": get_revenue_drop(curr_df)})

before_test = len(curr_df)
curr_df = curr_df[(curr_df['city'] != 'TEST') & (curr_df['customer_id'] != 'TEST')]
removed_test = before_test - len(curr_df)
steps.append({"Шаг": "5. Исключение TEST", "Строк осталось": len(curr_df), "Убрано строк": removed_test, "Падение выручки %": get_revenue_drop(curr_df)})

before_nan = len(curr_df)
curr_df = curr_df.dropna(subset=['amount', 'city'])
removed_nan = before_nan - len(curr_df)
steps.append({"Шаг": "6. Удаление NaN (amount, city)", "Строк осталось": len(curr_df), "Убрано строк": removed_nan, "Падение выручки %": get_revenue_drop(curr_df)})

journal_df = pd.DataFrame(steps)
display(journal_df)

,Шаг,Строк осталось,Убрано строк,Падение выручки %
0,0. Исходные данные,5463,0,-22.735039
1,1. Числовой формат amount,5463,0,-22.735039
2,2. Стандартизация городов,5463,0,-22.735039
3,3. Удаление дубликатов,5283,180,-19.593077
4,4. Преобразование дат,5283,0,-19.593077
5,5. Исключение TEST,5280,3,-2.337714
6,"6. Удаление NaN (amount, city)",5280,0,-2.337714


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [11]:
#4
missing_count = df_clean['delivery_days'].isna().sum()
print(f"Количество пропусков в delivery_days: {missing_count}\n")

print("Пропуски по городам:")
display(df_clean[df_clean['delivery_days'].isna()]['city'].value_counts())

print("\nПропуски по статусам заказа:")
display(df_clean[df_clean['delivery_days'].isna()]['status'].value_counts())

shymkent_df = df_clean[df_clean['city'] == 'Шымкент'].copy()

shymkent_dropna = shymkent_df.dropna(subset=['delivery_days'])
share_dropna = (shymkent_dropna['delivery_days'] == 2).mean() * 100

shymkent_median = shymkent_df.copy()
median_val = shymkent_df['delivery_days'].median()
shymkent_median['delivery_days'] = shymkent_median['delivery_days'].fillna(median_val)
share_median = (shymkent_median['delivery_days'] == 2).mean() * 100

shymkent_zero = shymkent_df.copy()
shymkent_zero['delivery_days'] = shymkent_zero['delivery_days'].fillna(0)
share_zero = (shymkent_zero['delivery_days'] == 2).mean() * 100

print(f"\nДоля доставки за 2 дня в Шымкенте:")
print(f"а) Через dropna: {share_dropna:.2f}%")
print(f"б) Через fillna(медиана={median_val}): {share_median:.2f}%")
print(f"в) Через fillna(0): {share_zero:.2f}%")

Количество пропусков в delivery_days: 158

Пропуски по городам:


city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64


Пропуски по статусам заказа:


status
оплачен    144
отменён     14
Name: count, dtype: int64


Доля доставки за 2 дня в Шымкенте:
а) Через dropna: 14.85%
б) Через fillna(медиана=5.0): 14.43%
в) Через fillna(0): 14.43%


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [12]:
import pandas as pd

df = pd.read_csv("orders.csv")

df = df.drop_duplicates()

city_mapping = {
    'алматы': 'Алматы', 'алмата': 'Алматы', 'almaty': 'Алматы',
    'астана': 'Астана', 'astana': 'Астана',
    'караганда': 'Караганда', 'karaganda': 'Караганда', 'крг': 'Караганда',
    'шымкент': 'Шымкент', 'shymkent': 'Шымкент',
    'актау': 'Актау', 'aktau': 'Актау',
    'атырау': 'Атырау', 'atyrau': 'Атырау'
}
df["city"] = df["city"].astype(str).str.strip().str.lower().map(city_mapping).fillna(df["city"])

df = df[df['city'] != 'TEST']

print(f"Количество уникальных городов: {df['city'].nunique()}")
print(f"Количество строк после удаления дубликатов: {len(df)}")

Количество уникальных городов: 6
Количество строк после удаления дубликатов: 5283


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
# 6
df_days = df_clean.copy()
df_days['day_of_week'] = df_days['order_date'].dt.day_name()

orders_by_day = df_days['day_of_week'].value_counts()
print("=== КОЛИЧЕСТВО ЗАКАЗОВ ПО ДНЯМ НЕДЕЛИ ===")
print(orders_by_day)

busiest_day = orders_by_day.idxmax()
busiest_count = orders_by_day.max()

quietest_day = orders_by_day.idxmin()
quietest_count = orders_by_day.min()

pct_diff = (busiest_count - quietest_count) / busiest_count * 100

print(f"\nСамый загруженный день: {busiest_day} ({busiest_count} зак.)")
print(f"Самый свободный день: {quietest_day} ({quietest_count} зак.)")
print(f"В самый свободный день заказов меньше на {pct_diff:.2f}% по сравнению с пиковым днем.")

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8